In [1]:
# Notebook for model comparison and evaluation
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [2]:
import pandas as pd
from src.data_ingestion import load_application_train, load_bureau, load_previous_application
from src.feature_engineering import engineer_features_pipeline, encode_categorical_features
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)

In [3]:
df_raw = load_application_train()
bureau_raw = load_bureau()
prev_app_raw = load_previous_application()

df = engineer_features_pipeline(df_raw, bureau_raw, prev_app_raw)

y = df['TARGET']
X = df.drop(columns=['TARGET', 'SK_ID_CURR'])
X_encoded = encode_categorical_features(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape, X_test_scaled.shape)

Loaded application_train: 307511 rows, 122 columns
Loaded bureau: 1716428 rows, 17 columns
Loaded previous_application: 1670214 rows, 37 columns
(246008, 249) (61503, 249)


In [4]:
from src.feature_engineering import prepare_model_data

X_train_scaled, X_test_scaled, y_train, y_test, scaler = prepare_model_data(df)
print(X_train_scaled.shape, X_test_scaled.shape)

(246008, 249) (61503, 249)


In [5]:
# Training the Random Forest model
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)

print("Random Forest trained successfully")

Random Forest trained successfully


In [6]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
# Evaluating the Random Forest model
y_pred_rf = rf_model.predict(X_test_scaled)
y_pred_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

print("Random Forest Results:")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba_rf))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

Random Forest Results:
Accuracy: 0.9181828528689657
Precision: 0.43920145190562615
Recall: 0.048741188318227595
F1 Score: 0.0877447425670776
ROC-AUC: 0.7400277089080567

Confusion Matrix:
[[56229   309]
 [ 4723   242]]


In [8]:
# Importing XGBoost and training the model
import xgboost
from xgboost import XGBClassifier
print(xgboost.__version__)

# scale_pos_weight = ratio of negative to positive class, XGBoost's balancing mechanism
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=100,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    n_jobs=-1
)

xgb_model.fit(X_train_scaled, y_train)

print("XGBoost trained successfully")

3.3.0
XGBoost trained successfully
